# [Lecture] ResNet: 딥러닝의 한계를 돌파한 잔차 학습 (Residual Learning)

> **강의 목표:** 모델이 깊어질 때 발생하는 문제를 이해하고, ResNet의 잔차 연결(Residual Connection)이 이를 어떻게 해결했는지 공학적으로 분석한다.

---

## 1. 서론: 모델이 깊어지면 무조건 좋은가?

### ⚠️ 기울기 저하 문제 (Degradation Problem)
단순히 레이어를 깊게 쌓는다고 해서 성능이 비례하여 좋아지지 않습니다. 

* **현상:** 특정 깊이 이상에서 오차가 다시 커지는 현상이 발생. (과적합이 아닌 **최적화 실패**의 문제)
* **원인:** 곱셈 기반의 연쇄 법칙(Chain Rule)으로 인해 기울기가 사라지거나 폭주하여 입력층 근처의 학습이 멈춤.



---

## 2. 핵심 철학: 잔차 학습 (Residual Learning)

기존의 모델이 직접적인 정답 $H(x)$를 찾으려 했다면, ResNet은 **입력과 출력의 차이(잔차)**를 학습하는 데 집중합니다.

### 🔢 수학적 정의
입력을 $x$, 학습해야 할 레이어를 $F(x)$라고 할 때, 최종 출력 $H(x)$는 다음과 같습니다.

$$H(x) = F(x) + x$$

* **잔차($F(x)$):** $$F(x) := H(x) - x$$
* **핵심:** $H(x)$를 처음부터 배우는 것보다, $F(x)$를 **0으로 수렴**시키는 것이 수학적으로 훨씬 최적화하기 쉽습니다.

---

## 3. 정보의 고속도로: 잔차 연결 (Residual Connection)

지름길(Shortcut/Skip Connection)을 통해 입력 데이터 $x$를 연산층 너머로 직접 전달합니다.

* **Identity Shortcut:** 입력과 출력의 규격이 같을 때 단순히 $x$를 더함.
* **Projection Shortcut:** 규격이 다를 때(Stride 사용 등) $1 \times 1$ Conv를 사용해 규격을 맞춤.



---

## 4. ResNet의 단계별 아키텍처 및 텐서 흐름

AI 엔지니어는 항상 **4차원 텐서 $(N, C, H, W)$**의 변화를 추적해야 합니다.

| 계층 (Layer) | 역할 및 공학적 의미 | 텐서 차원 예시 (ResNet-50) |
| :--- | :--- | :--- |
| **입력층 (Input)** | RGB 이미지 수용 | $(N, 3, 224, 224)$ |
| **합성곱 (Conv)** | 특징 추출 및 초기 해상도 축소 ($7 \times 7$) | $(N, 64, 112, 112)$ |
| **배치 정규화 (BN)** | 데이터 분포 정량화 및 학습 가속화 | 차원 유지 |
| **활성화 (ReLU)** | 비선형 임계치 주입 | 차원 유지 |
| **잔차 블록 (Block)** | 잔차 학습 수행 (Skip Connection의 핵심) | $(N, 256, 56, 56)$ |
| **평균 풀링 (AvgPool)** | 전역 평균 풀링 (공간 정보 압축) | $(N, 2048, 1, 1)$ |
| **완전연결 (FC)** | 1,000개 클래스 확률 매핑 | $(N, 1000)$ |

---

## 5. 고급 설계: 병목 블록 (Bottleneck Block)

ResNet-50 이상의 깊은 모델에서 연산 효율을 극대화하기 위한 설계입니다.

### ⏳ 모래시계형 구조 (1x1 → 3x3 → 1x1)
1. **$1 \times 1$ Conv (축소):** 채널 수를 줄여 연산량 급감.
2. **$3 \times 3$ Conv (연산):** 압축된 상태에서 특징을 효과적으로 학습.
3. **$1 \times 1$ Conv (확장):** 다시 원래 채널(또는 4배 확장)로 복구.



---

## 💡 결론

1. **지름길의 힘:** 잔차 연결은 깊은 층에서도 신호가 유실되지 않는 **'정보의 고속도로'**를 구축했습니다.
2. **범용성:** 이 철학은 오늘날 **Transformer** 등 모든 대규모 AI 아키텍처의 표준이 되었습니다.
3. **효율성:** 병목 블록을 통해 깊이는 깊어지면서도 파라미터 수는 경제적으로 관리할 수 있게 되었습니다.

---

In [ ]:
from torch import nn

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1):
        super().__init__()
        # 첫번째 컨볼루션 레이어 정의
        self.conv1 = nn.Conv2d(
            inplanes, planes,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )
        # 첫번째 배치 정규화 레이어 정의
        self.bn1 = nn.BatchNorm2d(planes)
        # ReLU 활성화 함수 정의
        self.relu = nn.ReLU(inplace=True)

        # 두번째 컨볼루션 레이어 정의
        self.conv2 = nn.Conv2d(
            planes, planes,
            kernel_size=3, stride=1,
            padding=1,
            bias=False
        )
        # 두번째 배치 정규화 레이어 정의
        self.bn2 = nn.BatchNorm2d(planes)

        ## 단축 연결 정의
        # 아무 처리가 없는 기본 레이어 정의
        self.shortcut = nn.Sequential()
        # 차원이 변경되는 경우 추가 컨볼루션 레이어 정의
        # 스트라이드가 1이 아니면 사이즈가 변경되고, 입력 채널과 출력채널이 다르면 이를 조정하는 컨볼루션 레이어가 필요합니다.
        if stride != 1 or inplanes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                # 단축 연결에서 차원 변경에 사용되는 레이어도 컨볼루션 레이어임을 알 수 있습니다.
                nn.Conv2d(
                    inplanes, self.expansion*planes,
                    kernel_size=1, stride=stride, bias=False
                ),
                # 단축 연결 연산도 마찬가지로 배치 정규화를 진행합니다.
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        # 첫번째 컨볼루션 레이어 전파
        out = self.conv1(x)
        # 첫번째 배치 정규화 레이어 전파
        out = self.bn1(out)
        # ReLU 활성화 함수 전파
        out = self.relu(out)
        # 두번째 컨볼루션 레이어 전파
        out = self.conv2(out)
        # 두번째 배치 정규화 레이어 전파
        out = self.bn2(out)
        # 단축 연결 더하기
        out += self.shortcut(x)
        # ReLU 활성화 함수 전파
        out = self.relu(out)
        return out


In [ ]:
import torch


class BottleneckBlock(nn.Module):
    # 컨볼루션 레이어 차원변경 배수
    expansion = 4

    def __init__(self, inplanes, planes, stride=1):
        super().__init__()
        
        # 첫번째 컨볼루션 레이어 정의
        self.conv1 = nn.Conv2d(
            inplanes, planes,
            kernel_size=1, bias=False
        )
        # 첫번째 배치 정규화 레이어 정의
        self.bn1 = nn.BatchNorm2d(planes)

        # 두번째 컨볼루션 레이어 정의
        self.conv2 = nn.Conv2d(
            planes, planes,
            kernel_size=3, stride=stride,
            padding=1,
            bias=False
        )
        # 두번째 배치 정규화 레이어 정의
        self.bn2 = nn.BatchNorm2d(planes)

        # 세번째 컨볼루션 레이어 정의
        self.conv3 = nn.Conv2d(
            planes, self.expansion*planes,
            kernel_size=1, bias=False
        )
        self.bn3 = nn.BatchNorm2d(self.expansion*planes)
        self.relu = nn.ReLU(inplace=True)

        ## 단축 연결 정의
        # 이전과 같습니다.
        self.shortcut = nn.Sequential()
        if stride != 1 or inplanes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    inplanes, self.expansion*planes,
                    kernel_size=1, stride=stride, bias=False
                ),
                nn.BatchNorm2d(self.expansion*planes)
            )

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)
        out = self.conv3(out)
        out = self.bn3(out)
        out += self.shortcut(x)
        out = self.relu(out)
        return out


In [ ]:

class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=1000):
        super().__init__()

        self.inplanes = 64

        # 이미지를 처리 가능한 피쳐맵으로 압축합니다.
        # 맥스풀링을 하는 이유는 가장 강한 신호만 남겨서 연산을 효율적으로 처리하기 위함입니다.
        self.stem = nn.Sequential(
            nn.Conv2d(3, self.inplanes, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(self.inplanes),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        ## 메인 영역입니다.
        # 각 스테이지는 여러개의 잔차 블록으로 구성됩니다.
        self.stage1 = self._make_layer(block, 64, layers[0], stride=1)
        self.stage2 = self._make_layer(block, 128, layers[1], stride=2)
        self.stage3 = self._make_layer(block, 256, layers[2], stride=2)
        self.stage4 = self._make_layer(block, 512, layers[3], stride=2)

        # 에버리지풀링을 하는 이유는 전역적인 고양이 정보를 유지한 채 연산 파라미터를 줄이기 위해서입니다. 채널당 1개의 평균값.
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        layers = []

        # 첫 번째 블록 생성
        layers.append(block(self.inplanes, planes, stride))
        self.inplanes = planes * block.expansion

        # 나머지 블록 생성
        for _ in range(num_blocks - 1):
            layers.append(block(self.inplanes, planes, 1))

        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.stem(x)
        out = self.stage1(out)
        out = self.stage2(out)
        out = self.stage3(out)
        out = self.stage4(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        out = self.fc(out)
        return out
        

In [ ]:
from torchvision import models
from torchinfo import summary


# ResNet-18 모델 생성: [2, 2, 2, 2]는 각 스테이지의 블록 개수, 1000은 출력 클래스 수
resnet18 = ResNet(BasicBlock, [2, 2, 2, 2], 1000)

# ResNet-34 모델 생성: [3, 4, 6, 3] 개의 BasicBlock 사용
resnet34 = ResNet(BasicBlock, [3, 4, 6, 3], 1000)

# ResNet-50 모델 생성: 여기서부터는 연산 효율을 위해 BottleneckBlock 사용
resnet50 = ResNet(BottleneckBlock, [3, 4, 6, 3], 1000)

# ResNet-101 모델 생성: 레이어 깊이를 조절하는 리스트 값이 변경됨
resnet101 = ResNet(BottleneckBlock, [3, 4, 23, 3], 1000)

# ResNet-152 모델 생성: 가장 깊은 구조의 모델
resnet152 = ResNet(BottleneckBlock, [3, 8, 36, 3], 1000)

# torchvision 라이브러리에서 사전 학습된(pretrained) ResNet-34 가중치를 불러옴
torch_model = models.resnet34(weights="ResNet34_Weights.IMAGENET1K_V1")

# torchinfo.summary를 통한 모델 구조 분석
# 입력 데이터: (1, 3, 224, 224) 크기의 4차원 텐서 (Batch, Channel, Height, Width)
resnet34_info = summary(resnet34, (1, 3, 224, 224), verbose=0)
torch_model_info = summary(torch_model, (1, 3, 224, 224), verbose=0)

# 두 모델의 전체 파라미터(Total Parameters) 개수 출력 및 비교
print(resnet34_info.total_params)
print(torch_model_info.total_params)

21797672
21797672
